# XpressLingua — Audio Render Pipeline (Colab)

Renders every phrase + sentence in `content/zh-manifest.json` to MP3 using open-source,
**commercially-licensed** TTS models.

**How to use:** Runtime → Change runtime type → **T4 GPU**, then Run All.
At the end a `zh-audio.zip` downloads — unzip it into the repo at `public/audio/zh/`.

Two engines:
- **Section A — MeloTTS** (MIT). Guaranteed to run, decent quality. Good first pass.
- **Section B — Qwen3-TTS** (Apache-2.0). Best open Mandarin tones. API may need a tweak —
  check the [model card](https://huggingface.co/Qwen/Qwen3-TTS-12Hz-1.7B-Base) if a cell errors.

Both write the SAME filenames, so whichever runs last wins. A Whisper pass at the end
flags any clip whose transcription drifts from the target text (hallucination filter).

In [ ]:
# Fetch the manifest straight from the repo
import json, urllib.request, os
MANIFEST_URL = 'https://raw.githubusercontent.com/Sup3rRookie/xpress-lingua/master/content/zh-manifest.json'
manifest = json.load(urllib.request.urlopen(MANIFEST_URL))
entries = manifest['entries']
os.makedirs('out', exist_ok=True)
print(len(entries), 'clips to render')

## Section A — MeloTTS (MIT, reliable)

In [ ]:
%pip install -q git+https://github.com/myshell-ai/MeloTTS.git
!python -m unidic download > /dev/null 2>&1 || true

In [ ]:
from melo.api import TTS
model = TTS(language='ZH', device='auto')
spk = model.hps.data.spk2id['ZH']
for i, e in enumerate(entries):
    wav_path = f"out/{e['id']}.wav"
    if os.path.exists(wav_path.replace('.wav', '.mp3')):
        continue  # resume-safe
    model.tts_to_file(e['text'], spk, wav_path, speed=0.9)
    if i % 20 == 0:
        print(f"{i+1}/{len(entries)} {e['id']}")
print('MeloTTS render done')

## Section B — Qwen3-TTS (Apache-2.0, best Mandarin — optional quality upgrade)

Skip this section if Section A output is good enough for now. If the API below
differs from your installed version, follow the model card examples — the loop
structure stays the same: text in → wav out at `out/<id>.wav`.

In [ ]:
RUN_QWEN = False  # flip to True for the quality upgrade
if RUN_QWEN:
    %pip install -q qwen-tts
    from qwen_tts import QwenTTS
    model_b = QwenTTS.from_pretrained('Qwen/Qwen3-TTS-12Hz-1.7B-Base', device='cuda')
    for i, e in enumerate(entries):
        model_b.tts_to_file(text=e['text'], language='zh', output_path=f"out/{e['id']}.wav")
        if i % 20 == 0:
            print(f"{i+1}/{len(entries)} {e['id']}")
    print('Qwen3-TTS render done')

## Convert to MP3 + Whisper sanity check + zip

In [ ]:
import subprocess, glob
for wav in glob.glob('out/*.wav'):
    mp3 = wav.replace('.wav', '.mp3')
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', wav, '-q:a', '4', mp3])
    os.remove(wav)
print(len(glob.glob('out/*.mp3')), 'mp3 files')

In [ ]:
# Hallucination filter: transcribe every clip, flag mismatches for manual listen
%pip install -q openai-whisper
import whisper, re
wm = whisper.load_model('small')
flags = []
strip = lambda s: re.sub(r'[^\u4e00-\u9fff]', '', s)
for e in entries:
    p = f"out/{e['id']}.mp3"
    if not os.path.exists(p):
        flags.append((e['id'], 'MISSING', ''))
        continue
    got = strip(wm.transcribe(p, language='zh')['text'])
    want = strip(e['text'])
    if want not in got and got not in want:
        flags.append((e['id'], want, got))
print(f'{len(flags)} flagged of {len(entries)} — listen to these before shipping:')
for f in flags: print(f)

In [ ]:
!cd out && zip -q -r ../zh-audio.zip .
from google.colab import files
files.download('zh-audio.zip')
print('Unzip into public/audio/zh/ in the repo, then run: node scripts/build-audio-manifest.js')